## Fabric IQ Accelerator Sample
        
### Create Ontology from Package

In [ ]:
# Install Fabric IQ Ontology Accelerator Package
%pip install /lakehouse/default/Files/fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl --q

In [ ]:
import sempy.fabric as fabric
import json
from fabricontology import create_ontology_item, generate_definition_from_package

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')

ontology_item_name = "AirlineOntology"
ontology_package_path = "/lakehouse/default/Files/airline_ontology_package.iq"

binding_lakehouse_name = ""
binding_lakehouse_schema_name = "dbo"  # replace this if using lakehouse without schemas
binding_eventhouse_name = ""
binding_eventhouse_cluster_uri = ""
binding_eventhouse_database_name = ""

items_df = fabric.list_items()
binding_lakehouse_item_id = str(items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == binding_lakehouse_name)].iloc[0].Id)
binding_eventhouse_item_id = str(items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == binding_eventhouse_name)].iloc[0].Id)
binding_workspace_id = workspace_id

ontology_definition, entity_types, relationship_types, data_bindings, contextualizations = generate_definition_from_package(
    ontology_package_path=ontology_package_path,
    ontology_name=ontology_item_name, 
    binding_workspace_id=binding_workspace_id,
    binding_lakehouse_item_id=binding_lakehouse_item_id,
    binding_lakehouse_schema_name=binding_lakehouse_schema_name,
    binding_eventhouse_item_id=binding_eventhouse_item_id,
    binding_eventhouse_cluster_uri=binding_eventhouse_cluster_uri,    
    binding_eventhouse_database_name=binding_eventhouse_database_name)

response = create_ontology_item(workspace_id=workspace_id, 
                           access_token=access_token,
                           ontology_item_name=ontology_item_name, 
                           ontology_definition=ontology_definition)
print(response.json())


In [ ]:
import sempy.fabric as fabric
import json
from fabricontology.generate_data import generate_instance_data, generate_events_data
from notebookutils import mssparkutils

ontology_package_path = "/lakehouse/default/Files/retail_ontology_package.iq"

# Create delta tables in the default lakehouse
lakehouse_schema = "dbo"  # replace this if using lakehouse without schemas.
response = generate_instance_data(spark, ontology_package_path=ontology_package_path, database=lakehouse_schema, mode="overwrite")

In [ ]:
# Create eventhouse tables 
eventhouse_cluster_uri = "https://trd-pkyux8h32yzu2v3vpn.z2.kusto.fabric.microsoft.com"
eventhouse_database = "EHRetail"
access_token=mssparkutils.credentials.getToken(eventhouse_cluster_uri)

response = generate_events_data(spark, 
        ontology_package_path=ontology_package_path,
        eventhouse_cluster_uri=eventhouse_cluster_uri,
        eventhouse_database=eventhouse_database,
        access_token=access_token )